# Stage 11 — the one FoR held-out run (D46)

This notebook carries out the D46 order end to end on a Colab T4 runtime (Protocol v1.0 §7; D31, D41, D45, D46, D57), from an exact clean commit, with the runner `src.evaluation.stage11_run` ([docs/stage11_run.md](../docs/stage11_run.md)) and the report `src.evaluation.stage11_report` ([docs/stage11_report.md](../docs/stage11_report.md)). It has two modes:

- `MODE = 'dry_run'` (committed default): environment gate, pytest, reproduction-gate caches, frozen artifacts, then the **rehearsal** on val-A and val-B (the D46 dry run, known answers), its evidence exported to Drive, then it **stops**. No claim, no held-out access. Rerunnable: each dry run keeps its own timestamped folder.
- `MODE = 'official'`: the same steps, then the registry **claim** → Drive flush and remount, claim verified on Drive → **score** → **report** → export → only then the results view. Runs once.

The notebook never names the held-out split: the runner fixes it in code (there is no split argument), and the guard `tests/governance/test_for_test_access.py` scans this file. Nothing is logged to an external service; the committed evidence is the record (D40). Run it only after the commit is reviewed and merged, and the official mode only after D57 is approved and PR B and PR C are merged.

## 1. Parameters

Before running, select the Colab runtime **T4 GPU** and, under "Version d'exécution / Runtime version", **2026.10** (D58 (1): the image that reproduces Stage 8 and 10, Python 3.13.16 and torch 2.11.0+cu130); the environment cell asserts both versions exactly. Replace `COMMIT` with the full 40-character approved commit. `MODE` is `'dry_run'` or `'official'`. The three versions name the frozen Drive folders the artifacts are copied from. `RESULTS_REF` is the repository path where the results PR will commit `metrics_report.md`; `claim` stores it and `complete` records it in the registry. There are no other knobs: batch size from `configs/cnn.yaml`, everything else from the committed evidence.

In [ ]:
REPO_URL = 'https://github.com/kahlanzzi/voice-spoofing-detection.git'
COMMIT = 'REPLACE_WITH_APPROVED_COMMIT'  # full 40-character SHA
MODE = 'dry_run'  # 'dry_run' or 'official'
CNN_VERSION = 'cnn-v1'
BASELINE_VERSION = 'baseline-v1'
RISK_VERSION = 'risk-v1'
DRIVE_ROOT = '/content/drive/MyDrive/voice-spoofing-detection'
RESULTS_REF = 'docs/results/stage11_for_test.md'
assert MODE in ('dry_run', 'official'), MODE

## 2. Mount Drive and refuse existing outputs

Drive layout under `processed/metadata/stage11/`:

| Path | Content |
|---|---|
| `run_registry.jsonl` (`REGISTRY`) | the live run registry (D46 (6)); written only by the runner's `claim` and `complete`, never deleted, moved or rewritten by this notebook |
| `run/` (`RUN_DEST`) | official mode: `runner/` (`CLAIM_OUT`, written in place by `claim` and `score`: claim and score evidence, `clip_scores/`), the metrics report, `rehearsal/`, the export manifest |
| `dry_run/<UTC timestamp>/` (`DRY_RUN_DEST`) | dry run: `rehearsal/` (written in place by `rehearse`) and the manifest, one folder per dry run |

Official mode stops here if `run/` exists; a dry run stops if its timestamped folder exists. The same checks run again right before each export.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys

from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path(DRIVE_ROOT)
STAGE11 = DRIVE / 'processed' / 'metadata' / 'stage11'
REGISTRY = STAGE11 / 'run_registry.jsonl'
RUN_DEST = STAGE11 / 'run'
DRY_RUN_STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
DRY_RUN_DEST = STAGE11 / 'dry_run' / DRY_RUN_STAMP


def refuse_existing_outputs():
    dest = RUN_DEST if MODE == 'official' else DRY_RUN_DEST
    if dest.exists():
        raise FileExistsError(f'{dest} already exists; Stage 11 outputs are never overwritten')


def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


refuse_existing_outputs()
assert (DRIVE / 'raw').is_dir(), f'{DRIVE / "raw"} not found; check DRIVE_ROOT'
print('MODE', MODE)

## 3. Check out the exact commit

The repository is cloned into `/content` and the full commit is checked out detached. HEAD must equal `COMMIT` exactly and the worktree must be clean. Every later output (caches, pytest XML, artifact copies, runner and report output) is written outside the clone, so the runner, which refuses a dirty worktree, sees a clean tree. `assert_clean` also refuses ignored files other than Python bytecode caches, because `.gitignore` hides some folders from `git status`.

In [ ]:
REPO = Path('/content/voice-spoofing-detection')
assert len(COMMIT) == 40, f'COMMIT must be the full 40-character SHA, got {COMMIT!r}'
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', 'checkout', '--detach', COMMIT], cwd=REPO, check=True)


def git(*args):
    return subprocess.run(['git', *args], cwd=REPO, check=True, capture_output=True, text=True).stdout


def assert_clean():
    head = git('rev-parse', 'HEAD').strip()
    assert head == COMMIT, (head, COMMIT)
    status = git('status', '--porcelain')
    assert not status, status
    ignored = [line[3:] for line in git('status', '--porcelain', '--ignored').splitlines() if line.startswith('!! ')]
    unexpected = [path for path in ignored if not path.rstrip('/').endswith('__pycache__')]
    assert not unexpected, unexpected


assert_clean()
os.chdir(REPO)
print('HEAD', COMMIT)

## 4. Environment gate (D46 (1), (2))

The runtime must be a Colab **T4** runtime (D46 (1): the runtime Stage 8 ran on and the only one that reproduced the Stage 8 val-A cache bytes, Entry 94). The GPU is **never used for scoring**: the runner scores every model on CPU, because the `risk-v1` policy was fitted on CPU logits, and no runner call takes a device argument. The GPU name is only asserted here and recorded in the export manifest.

The Colab runtime version must be **2026.10** ("Version d'exécution / Runtime version", selected with the T4 before running; D58 (1)): Python must be exactly 3.13.16 (D15, D58 (1)) and Colab's preinstalled torch exactly 2.11.0+cu130 (`requirements-cnn.txt`, D42, D58 (1)); any other image stops here. Torch is never installed by this notebook: only `requirements.txt` and `requirements-cnn.txt` are installed, and neither lists torch. The versions are then read again in a fresh process, so a silent upgrade during the install is caught. The runner asserts Python and torch again and records the full D46 (3) environment itself.

In [ ]:
import torch

GPU_NAME = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                          check=True, capture_output=True, text=True).stdout.strip()
assert 'T4' in GPU_NAME, f'D46 (1) requires a Colab T4 runtime, got {GPU_NAME!r}'
assert torch.__version__ == '2.11.0+cu130', f'D58 (1) requires Colab runtime 2026.10 (torch 2.11.0+cu130), got {torch.__version__}'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt', '-r', 'requirements-cnn.txt'],
               cwd=REPO, check=True)
PROBE = '''
import json, platform, numpy, torch
print(json.dumps({
    'python': platform.python_version(),
    'numpy': numpy.__version__,
    'torch': torch.__version__,
}))
'''
ENV = json.loads(subprocess.run([sys.executable, '-c', PROBE], check=True, capture_output=True, text=True).stdout)
assert ENV['torch'] == torch.__version__, (ENV['torch'], torch.__version__)
assert ENV['torch'] == '2.11.0+cu130', f"D58 (1) requires torch 2.11.0+cu130 (runtime 2026.10), got {ENV['torch']}"
assert ENV['python'] == '3.13.16', f"D58 (1) requires Python 3.13.16 (runtime 2026.10), got {ENV['python']}"
assert_clean()
for key, value in {**ENV, 'gpu_name': GPU_NAME}.items():
    print(f'{key}: {value}')

## 5. Verify and extract the pinned FoR archive

As in notebooks 03 to 05, the revision marker next to the Drive backup must equal the D21 FoR revision, read here from `configs/audit.yaml` (the D21 pin location). The archive is extracted with `filter='data'`. Its SHA-256 is computed and written to the export manifest as `for_archive_sha256_observed`: this is evidence only, never asserted or compared, and not a dataset pin (D21 unchanged).

In [ ]:
import tarfile
import yaml

EXPECTED_REVISION = yaml.safe_load((REPO / 'configs/audit.yaml').read_text(encoding='utf-8'))['for2sec']['revision']
ARCHIVE = DRIVE / 'raw' / 'for_2sec_ff8c82c7.tar'
REVISION_FILE = DRIVE / 'raw' / 'for_2sec_REVISION.txt'
assert REVISION_FILE.read_text(encoding='utf-8').strip() == EXPECTED_REVISION
DATA_ROOT = Path('/content/data')
AUDIO_ROOT = DATA_ROOT / 'for_2sec'
LOCAL_ARCHIVE = DATA_ROOT / 'for_2sec_ff8c82c7.tar'
DATA_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copy2(ARCHIVE, LOCAL_ARCHIVE)
if AUDIO_ROOT.exists():
    shutil.rmtree(AUDIO_ROOT)
with tarfile.open(LOCAL_ARCHIVE) as archive:
    archive.extractall(DATA_ROOT, filter='data')
FOR_ARCHIVE_SHA256_OBSERVED = sha256(LOCAL_ARCHIVE)  # recorded only (evidence, not a pin)
print('FoR archive extracted to', AUDIO_ROOT)
print('for_archive_sha256_observed:', FOR_ARCHIVE_SHA256_OBSERVED)

## 6. Run the test suite: no failure, error or skip (D46 (2))

`REQUIRE_TORCH_TESTS=1` turns a missing torch into a failure. The counts come from the JUnit XML written to `/content` (outside the clone), not from the terminal text, and every failure, error and skip stops the notebook.

In [ ]:
import xml.etree.ElementTree as ET


def stream(command, **kwargs):
    # Like subprocess.run(check=True), but prints the child's output line by line as it runs.
    env = {**kwargs.pop('env', os.environ), 'PYTHONUNBUFFERED': '1'}
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env,
                          **kwargs) as process:
        for line in process.stdout:
            print(line, end='')
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, command)


JUNIT = Path('/content/pytest_stage11.xml')
if JUNIT.exists():
    JUNIT.unlink()
assert_clean()
stream([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', f'--junitxml={JUNIT}'],
       cwd=REPO, env={**os.environ, 'REQUIRE_TORCH_TESTS': '1'})
root = ET.parse(JUNIT).getroot()
suites = [root] if root.tag == 'testsuite' else root.findall('testsuite')
COUNTS = {key: sum(int(suite.get(key, 0)) for suite in suites) for key in ('tests', 'failures', 'errors', 'skipped')}
print(COUNTS)
assert COUNTS['tests'] > 0, COUNTS
assert COUNTS['failures'] == COUNTS['errors'] == COUNTS['skipped'] == 0, COUNTS
assert_clean()

## 7. Reproduction-gate caches (D46 (4))

The runner reads two cache builds in the layout `<cache>/log_mel/<split>/` and `<cache>/mfcc/<split>/` (`--cache-a`, `--cache-b`; `src/evaluation/stage11_run.py`). Build A, `/content/cache_a`, holds val-A and val-B for both features (val-B is needed by the rehearsal); build B, `/content/cache_b`, holds val-A only (the gate rebuilds val-A only). Every sample hash is checked before decoding. Here the val-A files of both builds must be byte-identical, and every cache must record the F3 profile config SHA-256 and fingerprint stored in the repo (Stage 6 evidence, D39) and `COMMIT`. The runner then re-checks the log-Mel val-A `npz_sha256` against Stage 8 and the val-B one against Stage 10; the T4 MFCC is compared with the D58 CPU-runtime reference (step 8) within the D58 (3)(b) tolerance instead of with Stage 7's bytes, which are recorded only (D58 (3)). Only val-A and val-B are ever built; no cache is built for the training or held-out splits.

In [ ]:
MANIFEST = REPO / 'data/manifests/for2sec_frozen_manifest.csv'
PROFILE = REPO / 'configs/preprocessing.yaml'
F3 = json.loads((REPO / 'docs/audit_evidence/stage6_preprocessing/profile_report.json').read_text(encoding='utf-8'))['profile']
assert sha256(PROFILE) == F3['config_sha256'], 'configs/preprocessing.yaml differs from F3'
FEATURE_MODULES = {'log_mel': 'src.features.logmel_cache', 'mfcc': 'src.features.mfcc_cache'}
CACHE_A, CACHE_B = Path('/content/cache_a'), Path('/content/cache_b')
CACHE_PLAN = ((CACHE_A, ('val-A', 'val-B')), (CACHE_B, ('val-A',)))
for cache, splits in CACHE_PLAN:
    if cache.exists():
        shutil.rmtree(cache)
    cache.mkdir()
    for feature, module in FEATURE_MODULES.items():
        for split in splits:
            assert_clean()
            subprocess.run([
                sys.executable, '-m', module,
                '--root', str(AUDIO_ROOT), '--manifest', str(MANIFEST), '--profile', str(PROFILE),
                '--split', split, '--out', str(cache / feature / split),
            ], cwd=REPO, check=True, stdout=subprocess.DEVNULL)


def file_hashes(folder):
    return {p.relative_to(folder).as_posix(): sha256(p) for p in sorted(folder.rglob('*')) if p.is_file()}


CACHE_HASHES_A, CACHE_HASHES_B = file_hashes(CACHE_A), file_hashes(CACHE_B)
for cache, splits in CACHE_PLAN:
    built = sorted(file_hashes(cache))
    planned = sorted(f'{f}/{s}/{s}.{ext}' for f in FEATURE_MODULES for s in splits for ext in ('npz', 'json'))
    assert built == planned, (cache, built)
CACHE_A_VAL_A = {rel: digest for rel, digest in CACHE_HASHES_A.items() if rel.split('/')[1] == 'val-A'}
assert CACHE_HASHES_B == CACHE_A_VAL_A, 'the two val-A cache builds differ'
for cache, splits in CACHE_PLAN:
    for feature in FEATURE_MODULES:
        for split in splits:
            meta = json.loads((cache / feature / split / f'{split}.json').read_text(encoding='utf-8'))
            assert meta['profile_config_sha256'] == F3['config_sha256'], (cache, feature, split)
            assert meta['profile_fingerprint'] == F3['fingerprint'], (cache, feature, split)
            assert meta['code_commit'] == COMMIT, (cache, feature, split)
assert_clean()
CACHE_HASHES_A

## 8. Copy the frozen artifacts from Drive

Copied to `/content/stage11_artifacts/`, outside the clone: the three Stage 8 checkpoints `checkpoints/cnn/<CNN_VERSION>/seed_0..2/`, the Stage 7 baseline artifact `checkpoints/baseline/<BASELINE_VERSION>/`, the Stage 10 `checkpoints/risk/<RISK_VERSION>/risk_policy.json` and `val_b_scores.npz`, and the D58 MFCC reference `processed/metadata/stage11/mfcc_reference/{val-A,val-B}/` (the Stage 7 val-A and val-B MFCC caches rebuilt on a Colab CPU runtime, D58 (3)(a)) to `mfcc_reference/`, passed to `rehearse`, `claim` and `score` as `--mfcc-reference`. Nothing is verified or loaded here: the runner checks every hash against the committed Stage 7, 8 and 10 evidence (the reference's `npz_sha256` against Stage 7's) before it reads any cache. The SHA-256 of the reference files is only recorded (`MFCC_REFERENCE_SHA256`, in the export manifest).

In [ ]:
ARTIFACTS = Path('/content/stage11_artifacts')
CHECKPOINTS_LOCAL = ARTIFACTS / 'checkpoints'
BASELINE_LOCAL = ARTIFACTS / 'baseline' / BASELINE_VERSION
RISK_LOCAL = ARTIFACTS / 'risk' / RISK_VERSION
MFCC_REFERENCE_LOCAL = ARTIFACTS / 'mfcc_reference'
CNN_DRIVE = DRIVE / 'checkpoints' / 'cnn' / CNN_VERSION
BASELINE_DRIVE = DRIVE / 'checkpoints' / 'baseline' / BASELINE_VERSION
RISK_DRIVE = DRIVE / 'checkpoints' / 'risk' / RISK_VERSION
MFCC_REFERENCE_DRIVE = STAGE11 / 'mfcc_reference'
if ARTIFACTS.exists():
    shutil.rmtree(ARTIFACTS)
for seed in (0, 1, 2):
    assert (CNN_DRIVE / f'seed_{seed}').is_dir(), f'{CNN_DRIVE / f"seed_{seed}"} not found'
    shutil.copytree(CNN_DRIVE / f'seed_{seed}', CHECKPOINTS_LOCAL / f'seed_{seed}')
assert BASELINE_DRIVE.is_dir(), f'{BASELINE_DRIVE} not found'
shutil.copytree(BASELINE_DRIVE, BASELINE_LOCAL)
RISK_LOCAL.mkdir(parents=True)
for name in ('risk_policy.json', 'val_b_scores.npz'):
    shutil.copy2(RISK_DRIVE / name, RISK_LOCAL / name)
for split in ('val-A', 'val-B'):  # D58 (3)(a); checked against Stage 7 by the runner, not here
    assert (MFCC_REFERENCE_DRIVE / split).is_dir(), f'{MFCC_REFERENCE_DRIVE / split} not found'
    shutil.copytree(MFCC_REFERENCE_DRIVE / split, MFCC_REFERENCE_LOCAL / split)
MFCC_REFERENCE_SHA256 = file_hashes(MFCC_REFERENCE_LOCAL)  # recorded only
assert not ARTIFACTS.resolve().is_relative_to(REPO.resolve()), ARTIFACTS
for path in sorted(ARTIFACTS.rglob('*')):
    if path.is_file():
        print(path.relative_to(ARTIFACTS).as_posix())

## 9. Rehearsal (the D46 dry run)

`rehearse` runs the same pre-claim checks as the official run (environment gate, evidence, checkpoint, baseline and policy hashes, the reproduction gate on `cache_a` and `cache_b`), then scores val-A and val-B with the same code path and checks known answers: in-memory features equal the verified caches, each seed's and the baseline's val-A EER, the Stage 10 bundle-seed val-B scores and `p` byte for byte, the baseline val-B EER ([docs/stage11_run.md](../docs/stage11_run.md), "Rehearsal"). It opens no registry and reads only val-A and val-B. It writes `rehearsal_report.json` and `pip_freeze.txt` into a new folder: in dry-run mode `dry_run/<UTC timestamp>/rehearsal/` on Drive (the folder `dry_run/<UTC timestamp>/` is created here with `exist_ok=False`), so the dry run also proves that the runner's folder creation and exclusive writes work on Drive, as the official run needs; in official mode the local `/content/stage11_rehearsal`, copied to `run/rehearsal/` at export.

If the runner raises, the notebook stops here; the runner has already written the failed report (`"passed": false`, `failed_check`, the error) into that folder for the team. A rehearsal that ends with a failed check (`failed_check` not null) stops Stage 11 until a logged decision; one that crashes on an infrastructure error (`failed_check` null) may be rerun from the same COMMIT (D46 (8)). Only `passed` is read from the report.

In [ ]:
RUNNER = 'src.evaluation.stage11_run'
INPUTS = [
    '--checkpoints', str(CHECKPOINTS_LOCAL), '--baseline', str(BASELINE_LOCAL),
    '--risk-policy', str(RISK_LOCAL / 'risk_policy.json'), '--profile', str(PROFILE),
    '--config', str(REPO / 'configs/cnn.yaml'), '--manifest', str(MANIFEST),
    '--stage7-evidence', str(REPO / 'docs/audit_evidence/stage7_baseline'),
    '--stage8-evidence', str(REPO / 'docs/audit_evidence/stage8_cnn'),
    '--stage10-evidence', str(REPO / 'docs/audit_evidence/stage10_risk'),
    '--cache-a', str(CACHE_A), '--cache-b', str(CACHE_B), '--mfcc-reference', str(MFCC_REFERENCE_LOCAL),
]
if MODE == 'dry_run':
    refuse_existing_outputs()
    DRY_RUN_DEST.mkdir(parents=True, exist_ok=False)
    REHEARSAL_OUT = DRY_RUN_DEST / 'rehearsal'  # on Drive; the runner creates it
else:
    REHEARSAL_OUT = Path('/content/stage11_rehearsal')
if REHEARSAL_OUT.exists():
    raise FileExistsError(f'{REHEARSAL_OUT} already exists; the rehearsal writes a new folder')
assert_clean()
stream([sys.executable, '-m', RUNNER, 'rehearse', '--root', str(AUDIO_ROOT), *INPUTS,
        '--stage10-scores', str(RISK_LOCAL / 'val_b_scores.npz'), '--out', str(REHEARSAL_OUT)], cwd=REPO)
assert_clean()
REHEARSAL_PASSED = json.loads((REHEARSAL_OUT / 'rehearsal_report.json').read_text(encoding='utf-8'))['passed']
assert REHEARSAL_PASSED is True, 'the rehearsal did not pass; Stage 11 stops (D46 (8))'
assert sorted(p.name for p in REHEARSAL_OUT.iterdir()) == ['pip_freeze.txt', 'rehearsal_report.json']
print('Rehearsal passed.')

## 10. Dry-run export, then stop

**Dry-run mode only.** The rehearsal already wrote `rehearsal_report.json` and `pip_freeze.txt` to `processed/metadata/stage11/dry_run/<UTC timestamp>/rehearsal/` on Drive; they are hashed there, in place, and `stage11_dry_run_manifest.json` is written next to `rehearsal/`. The manifest records the commit, versions, environment, GPU name, pytest counts, the SHA-256 of every cache file of both builds and of the D58 MFCC reference files, the F3 values, the D21 FoR revision and the observed archive SHA-256 (evidence only), and each exported file's SHA-256. Drive is then flushed, unmounted and remounted, and every file is hashed again against the manifest (a missing, extra or changed file fails).

In official mode this cell only defines the export helpers: the rehearsal evidence is exported later with the run, under `run/rehearsal/` (step 15). A dry run ends here; the next cell refuses to run in dry-run mode.

In [ ]:
def drive_entry(path):
    # Manifest entry of a file already on Drive, hashed in place.
    return {'path': path.relative_to(DRIVE).as_posix(), 'size': path.stat().st_size, 'sha256': sha256(path)}


def export_file(source, dest):
    # Copy one file to Drive, refusing to overwrite, and return its manifest entry.
    assert not dest.exists(), f'{dest} already exists'
    shutil.copy2(source, dest)
    entry = drive_entry(dest)
    assert entry['sha256'] == sha256(source), f'{dest} differs from {source}'
    return entry


def verify_after_remount(root, manifest):
    # Every file under root on Drive must match the manifest: no missing, extra or changed file.
    expected = {entry['path']: entry['sha256'] for entry in manifest['files']}
    stored = {p.relative_to(DRIVE).as_posix() for p in root.rglob('*') if p.is_file()}
    missing, extra = sorted(set(expected) - stored), sorted(stored - set(expected))
    assert not missing and not extra, f'missing on Drive: {missing}; extra on Drive: {extra}'
    mismatched = [path for path, digest in sorted(expected.items()) if sha256(DRIVE / path) != digest]
    assert not mismatched, f'SHA-256 mismatch on Drive after remount: {mismatched}'
    return len(expected)


def base_manifest():
    return {
        'commit': COMMIT,
        'mode': MODE,
        'cnn_version': CNN_VERSION,
        'baseline_version': BASELINE_VERSION,
        'risk_version': RISK_VERSION,
        'python': ENV['python'],
        'numpy': ENV['numpy'],
        'torch': ENV['torch'],
        'gpu_name': GPU_NAME,
        'created_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
        'for_revision': EXPECTED_REVISION,
        'for_archive_sha256_observed': FOR_ARCHIVE_SHA256_OBSERVED,
        'pytest_counts': COUNTS,
        'cache_sha256': {'cache_a': CACHE_HASHES_A, 'cache_b': CACHE_HASHES_B},
        'mfcc_reference_sha256': MFCC_REFERENCE_SHA256,
        'f3': {'config_sha256': F3['config_sha256'], 'fingerprint': F3['fingerprint']},
        'rehearsal_passed': REHEARSAL_PASSED,
    }


if MODE == 'dry_run':
    assert REHEARSAL_OUT == DRY_RUN_DEST / 'rehearsal', REHEARSAL_OUT
    stored = sorted(p.relative_to(DRY_RUN_DEST).as_posix() for p in DRY_RUN_DEST.rglob('*') if p.is_file())
    assert stored == ['rehearsal/pip_freeze.txt', 'rehearsal/rehearsal_report.json'], stored
    files = [drive_entry(DRY_RUN_DEST / rel) for rel in stored]
    DRY_RUN_MANIFEST = {**base_manifest(), 'files': files}
    DRY_RUN_MANIFEST_PATH = DRY_RUN_DEST / 'stage11_dry_run_manifest.json'
    assert not DRY_RUN_MANIFEST_PATH.exists(), DRY_RUN_MANIFEST_PATH
    DRY_RUN_MANIFEST_TEXT = json.dumps(DRY_RUN_MANIFEST, indent=2, sort_keys=True) + '\n'
    DRY_RUN_MANIFEST_PATH.write_text(DRY_RUN_MANIFEST_TEXT, encoding='utf-8')
    print(DRY_RUN_MANIFEST_TEXT, end='')
    drive.flush_and_unmount()
    drive.mount('/content/drive', force_remount=True)
    checked = {'files': [*DRY_RUN_MANIFEST['files'], {
        'path': DRY_RUN_MANIFEST_PATH.relative_to(DRIVE).as_posix(),
        'sha256': hashlib.sha256(DRY_RUN_MANIFEST_TEXT.encode('utf-8')).hexdigest()}]}
    print(f'Drive verified after remount: {verify_after_remount(DRY_RUN_DEST, checked)} files match the manifest.')
    print('Dry run complete. Stop here.')

## 11. Claim (D46 (5)) — official mode only

From here on every code cell starts with `assert MODE == 'official'`, so a dry run cannot go further.

`refuse_existing_outputs()` checks right before the claim that `run/` (`RUN_DEST`) does not exist. The notebook does not create it: `claim` re-runs every pre-claim check (nothing is written if one fails), then appends the `started` record to the live registry on Drive (`REGISTRY`), re-reads it, and only then creates `run/runner/` (`CLAIM_OUT`, with `run/`) by an exclusive `mkdir` and writes `claim.json`, `environment.json` and `pip_freeze.txt` into it. So a failure before the claim leaves no folder behind, and a permitted pre-claim rerun is not blocked by an empty `run/`. It reads no audio. The SHA-256 of the three files is recorded right away (`CLAIM_HASHES`) and re-checked after the flush and remount.

The runner writes straight to Drive, not to `/content`, because the `scoring_started.json` marker that `score` creates in this folder must survive a runtime crash, so that the no-rerun rule holds in code and not only by procedure ([docs/stage11_run.md](../docs/stage11_run.md) Why (l)).

**Rerun rule (D46 (8), pre-declared): before the claim, an infrastructure failure (Drive, clone, pip, archive) may be rerun from the same COMMIT; an environment-gate or reproduction-gate failure, or a rehearsal that ends with a failed check (failed_check not null), stops Stage 11 until a logged decision; a rehearsal that crashes on an infrastructure error (failed_check null) may be rerun from the same COMMIT. After the claim, there is no rerun ever without a new decision, including a crash before any score exists; everything written stays for team review. Nothing in this notebook reads or reacts to the test results before they are exported.**

In [ ]:
assert MODE == 'official', 'dry run: stop before the claim'
CLAIM_OUT = RUN_DEST / 'runner'
refuse_existing_outputs()
assert_clean()
stream([sys.executable, '-m', RUNNER, 'claim', *INPUTS, '--registry', str(REGISTRY),
        '--results-ref', RESULTS_REF, '--out', str(CLAIM_OUT)], cwd=REPO)
assert_clean()
CLAIM_FILES = ('claim.json', 'environment.json', 'pip_freeze.txt')
assert sorted(p.name for p in CLAIM_OUT.iterdir()) == sorted(CLAIM_FILES)
CLAIM_HASHES = {name: sha256(CLAIM_OUT / name) for name in CLAIM_FILES}
print('Claim written to', CLAIM_OUT)

## 12. Drive flush, remount and verify the claim (D46 (6))

On Colab, Drive writes go to a local cache first; reading the registry on the same mount would read that cache. Drive is therefore flushed, unmounted and remounted, and the claim is checked on what Drive actually stored: the registry's SHA-256 must equal `registry_sha256` in `claim.json`, its last line must be this run's `started` record, and the three claim files in `run/runner/` must still match `CLAIM_HASHES`. `RUN_ID` is read from `claim.json` in code; it is never typed into this notebook. Any failure here happens after the claim: stop, keep everything, and report (D46 (8)).

In [ ]:
assert MODE == 'official', 'dry run: stop before the claim'
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)
CLAIM = json.loads((CLAIM_OUT / 'claim.json').read_text(encoding='utf-8'))
RUN_ID = CLAIM['run_id']
assert CLAIM['code_commit'] == COMMIT, CLAIM['code_commit']
assert CLAIM['results_ref'] == RESULTS_REF, CLAIM['results_ref']
assert sha256(REGISTRY) == CLAIM['registry_sha256'], 'the registry on Drive is not the one the claim recorded'
LAST = json.loads(REGISTRY.read_text(encoding='utf-8').splitlines()[-1])
assert LAST['run_id'] == RUN_ID and LAST['status'] == 'started', (LAST['run_id'], LAST['status'])
assert LAST['code_commit'] == COMMIT and LAST['split'] == CLAIM['split'], LAST
assert {name: sha256(CLAIM_OUT / name) for name in CLAIM_FILES} == CLAIM_HASHES, 'claim files changed on Drive'
print('Claim verified on Drive after remount: run_id', RUN_ID)

## 13. Score

`score` takes the claim's folder on Drive (`CLAIM_OUT`, `run/runner/`), `RUN_ID` from `claim.json`, the Drive registry and the same inputs as the claim, plus the FoR root. It checks `claim.json` against `RUN_ID` and the commit, creates the once-only marker `scoring_started.json`, records the scoring session's environment, re-runs every check (including the reproduction gate on the val-A caches still in `/content`), checks the registry bytes against the claim, then computes the features twice in memory (never cached, D46 (7)), scores all four models twice on CPU, writes `clip_scores/` and `run_report.json`, and records `complete` in the registry. Afterwards only identity is checked: `run_report.json` records `COMMIT` and `RUN_ID`, and the registry's last line is this run's `completed` record. Nothing is read from the scores.

In [ ]:
assert MODE == 'official', 'dry run: stop before the claim'
assert_clean()
stream([sys.executable, '-m', RUNNER, 'score', '--root', str(AUDIO_ROOT), *INPUTS, '--registry', str(REGISTRY),
        '--run-id', RUN_ID, '--out', str(CLAIM_OUT)], cwd=REPO)
assert_clean()
RUN_REPORT = json.loads((CLAIM_OUT / 'run_report.json').read_text(encoding='utf-8'))
assert RUN_REPORT['code_commit'] == COMMIT, RUN_REPORT['code_commit']
assert RUN_REPORT['run_id'] == RUN_ID, RUN_REPORT['run_id']
DONE = json.loads(REGISTRY.read_text(encoding='utf-8').splitlines()[-1])
assert DONE['run_id'] == RUN_ID and DONE['status'] == 'completed', (DONE['run_id'], DONE['status'])
assert DONE['results_ref'] == RESULTS_REF, DONE['results_ref']
print('Scored and completed: run_id', RUN_ID)

## 14. Report (PR B)

`src.evaluation.stage11_report` reads only the stored run files (`run/runner/` on Drive), the registry, the F2 manifest, the frozen policy and the Stage 7 / Stage 10 evidence; it runs its integrity checks, computes the metrics under D57 and writes `metrics_report.json` and `metrics_report.md` into `/content/stage11_report` (which must not exist). It prints only the two paths. This notebook does not open, print or branch on any metric before step 16.

In [ ]:
assert MODE == 'official', 'dry run: stop before the claim'
REPORT_OUT = Path('/content/stage11_report')
if REPORT_OUT.exists():
    raise FileExistsError(f'{REPORT_OUT} already exists')
assert_clean()
stream([sys.executable, '-m', 'src.evaluation.stage11_report', '--run-dir', str(CLAIM_OUT),
        '--registry', str(REGISTRY), '--manifest', str(MANIFEST),
        '--risk-policy', str(RISK_LOCAL / 'risk_policy.json'),
        '--stage7-evidence', str(REPO / 'docs/audit_evidence/stage7_baseline'),
        '--stage10-evidence', str(REPO / 'docs/audit_evidence/stage10_risk'), '--out', str(REPORT_OUT)], cwd=REPO)
assert_clean()
assert sorted(p.name for p in REPORT_OUT.iterdir()) == ['metrics_report.json', 'metrics_report.md']

## 15. Official export, flush, remount and re-verify

The runner's files are already on Drive in `run/runner/` and are not copied: the folder must hold exactly the three claim files, `scoring_started.json`, `environment_score.json`, `run_report.json` and `clip_scores/` with its 8 files, the claim files must still match `CLAIM_HASHES`, and every file is hashed in place into the manifest. Copied to `run/` (`rehearsal/` created with `exist_ok=False`, no file overwritten): `metrics_report.json` and `metrics_report.md`, and the rehearsal evidence under `rehearsal/`; each copy is re-hashed against its `/content` source. `stage11_export_manifest.json` records what notebook 05's manifest does, plus the GPU name, `run_id`, the rehearsal's `passed`, and the registry's SHA-256 and line count after `complete` (harness "Run registry" (e)). Drive is then flushed, unmounted and remounted; every file under `run/` is hashed again against the manifest (a missing, extra or changed file fails) and the registry against its recorded SHA-256. These are candidate records until human review.

In [ ]:
assert MODE == 'official', 'dry run: stop before the claim'
SCORE_FILES = ('scoring_started.json', 'environment_score.json', 'run_report.json')
runner_files = sorted(p.relative_to(CLAIM_OUT).as_posix() for p in CLAIM_OUT.rglob('*') if p.is_file())
clip_files = sorted(p.name for p in (CLAIM_OUT / 'clip_scores').iterdir())
assert len(clip_files) == 8, clip_files
assert runner_files == sorted([*CLAIM_FILES, *SCORE_FILES, *(f'clip_scores/{name}' for name in clip_files)]), runner_files
assert {name: sha256(CLAIM_OUT / name) for name in CLAIM_FILES} == CLAIM_HASHES, 'claim files changed on Drive'
assert sorted(p.name for p in RUN_DEST.iterdir()) == ['runner'], sorted(p.name for p in RUN_DEST.iterdir())
(RUN_DEST / 'rehearsal').mkdir(exist_ok=False)
files = [drive_entry(CLAIM_OUT / rel) for rel in runner_files]
files += [export_file(REPORT_OUT / name, RUN_DEST / name) for name in ('metrics_report.json', 'metrics_report.md')]
files += [export_file(REHEARSAL_OUT / name, RUN_DEST / 'rehearsal' / name)
          for name in ('rehearsal_report.json', 'pip_freeze.txt')]
REGISTRY_SHA256 = sha256(REGISTRY)
REGISTRY_LINES = len(REGISTRY.read_text(encoding='utf-8').splitlines())
EXPORT_MANIFEST = {
    **base_manifest(),
    'run_id': RUN_ID,
    'results_ref': RESULTS_REF,
    'registry': {'path': REGISTRY.relative_to(DRIVE).as_posix(), 'sha256': REGISTRY_SHA256,
                 'lines': REGISTRY_LINES},
    'files': files,
}
MANIFEST_PATH = RUN_DEST / 'stage11_export_manifest.json'
MANIFEST_TEXT = json.dumps(EXPORT_MANIFEST, indent=2, sort_keys=True) + '\n'
MANIFEST_PATH.write_text(MANIFEST_TEXT, encoding='utf-8')
print(MANIFEST_TEXT, end='')
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)
checked = {'files': [*EXPORT_MANIFEST['files'], {
    'path': MANIFEST_PATH.relative_to(DRIVE).as_posix(),
    'sha256': hashlib.sha256(MANIFEST_TEXT.encode('utf-8')).hexdigest()}]}
print(f'Drive verified after remount: {verify_after_remount(RUN_DEST, checked)} files match the manifest.')
assert sha256(REGISTRY) == REGISTRY_SHA256, 'the registry changed after the export'
print('registry_sha256:', REGISTRY_SHA256, 'lines:', REGISTRY_LINES)

## 16. Results view

Only now, after the run is completed in the registry and every file is exported and verified on Drive, is the report shown. The numbers are reported whatever the outcome; nothing in Stage 11 changes because of them: no threshold, model, policy or reporting convention is revised (D57, Protocol §12).

In [ ]:
assert MODE == 'official', 'dry run: stop before the claim'
from IPython.display import Markdown, display

display(Markdown((REPORT_OUT / 'metrics_report.md').read_text(encoding='utf-8')))

## 17. What is committed

After human review, a results PR commits:

- `docs/audit_evidence/run_registry.jsonl`: a byte copy of the Drive registry, with its SHA-256 and line count (D46 (6); they must equal `registry` in `stage11_export_manifest.json`);
- under `docs/audit_evidence/stage11_run/`: the claim and score JSON evidence (`claim.json`, `environment.json`, `pip_freeze.txt`, `scoring_started.json`, `environment_score.json`, `run_report.json`), `rehearsal_report.json`, `metrics_report.json` and `stage11_export_manifest.json`;
- `metrics_report.md` at `RESULTS_REF` (`docs/results/stage11_for_test.md`).

The per-clip CSVs and sidecars stay on Drive only (`run/runner/clip_scores/`), bound by their SHA-256 in `run_report.json`, as Stage 10's `val_b_scores.npz` (AGENTS.md rule 7); Stages 14 and 16 read them there (D41, D43).